# Спайк 1 — Pipecat Smart Turn v3 (аудио-модель)

Семантический VAD: смотрит на **сырое аудио** (не транскрипт) и говорит,
закончил ли человек реплику.

- Модель: Whisper Tiny encoder + linear head, ~8M параметров
- Вход: PCM 16kHz mono, до 8 секунд (берётся хвост реплики)
- Выход: вероятность `complete` (0..1), уже после sigmoid
- 23 языка включая русский
- HF: https://huggingface.co/pipecat-ai/smart-turn-v3

Запускаем через `onnxruntime` — torch не нужен.

## Установка зависимостей

In [1]:
# !pip install onnxruntime transformers huggingface_hub librosa numpy
# для генерации тестовой речи (опционально): !pip install gtts soundfile

## Загрузка модели

In [2]:
import numpy as np
import onnxruntime as ort
from transformers import WhisperFeatureExtractor
from huggingface_hub import hf_hub_download

# варианты: smart-turn-v3.2-cpu.onnx / smart-turn-v3.2-gpu.onnx
MODEL_FILE = "smart-turn-v3.2-cpu.onnx"
onnx_path = hf_hub_download("pipecat-ai/smart-turn-v3", MODEL_FILE)

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
feature_extractor = WhisperFeatureExtractor(chunk_length=8)  # 8 сек контекста

print("inputs :", [(i.name, i.shape) for i in session.get_inputs()])
print("outputs:", [(o.name, o.shape) for o in session.get_outputs()])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


inputs : [('input_features', ['s6', 80, 800])]
outputs: [('logits', ['s6', 1])]


## Функция предсказания

In [3]:
def predict_endpoint(audio: np.ndarray, sr: int = 16000) -> float:
    """Возвращает вероятность того, что реплика завершена (0..1).

    audio — моно float32 в диапазоне [-1, 1]. Если sr != 16000, ресемплим.
    """
    import librosa
    audio = audio.astype(np.float32)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    # нормализация в [-1, 1]
    peak = np.max(np.abs(audio)) if audio.size else 1.0
    if peak > 1.0:
        audio = audio / peak
    # берём последние 8 секунд (где заканчивается реплика)
    if len(audio) > 8 * 16000:
        audio = audio[-8 * 16000:]

    feats = feature_extractor(
        audio, sampling_rate=16000, return_tensors="np",
        padding="max_length", max_length=8 * 16000,
        truncation=True, do_normalize=True,
    )
    x = feats.input_features.squeeze(0).astype(np.float32)[None]   # [1, 80, 800]
    out = session.run(None, {"input_features": x})[0]
    score = float(out[0][0])                                       # уже sigmoid
    return score

## Смоук-тест (модель запускается)

In [4]:
dummy = (np.random.randn(16000) * 0.1).astype(np.float32)  # 1 сек шума
print("smoke score:", round(predict_endpoint(dummy), 4))

smoke score: 0.9854


## Тест на реальной русской речи (gTTS)

Генерируем несколько фраз через Google TTS.

> Важно: TTS читает **любую** фразу с завершающей интонацией, поэтому
> `incomplete` примеры будут получать завышенный score. Для честной оценки
> нужны реальные записи звонков с оборванными репликами. Здесь — только
> проверка что пайплайн работает на живом аудио.

In [5]:
import os
try:
    from gtts import gTTS
    import librosa
    samples = {
        "complete_1":   "Здравствуйте, я хотел бы узнать баланс по своей карте.",
        "complete_2":   "Спасибо большое, до свидания.",
        "incomplete_1": "Мне нужно",
        "incomplete_2": "А скажите пожалуйста какой у меня",
    }
    os.makedirs("/tmp/tts", exist_ok=True)
    print(f"{'sample':<16}{'score':>8}  expected")
    for name, text in samples.items():
        path = f"/tmp/tts/{name}.mp3"
        if not os.path.exists(path):
            gTTS(text, lang="ru").save(path)
        audio, _ = librosa.load(path, sr=16000, mono=True)
        exp = "complete" if name.startswith("complete") else "incomplete"
        print(f"{name:<16}{predict_endpoint(audio):>8.4f}  {exp}")
except Exception as e:
    print("gTTS недоступен, пропускаем:", e)

sample             score  expected


complete_1        0.9425  complete
complete_2        0.9256  complete
incomplete_1      0.8548  incomplete
incomplete_2      0.7858  incomplete


## Замер скорости инференса

In [6]:
import time
audio = (np.random.randn(8 * 16000) * 0.1).astype(np.float32)
predict_endpoint(audio)  # прогрев
t0 = time.perf_counter()
N = 50
for _ in range(N):
    predict_endpoint(audio)
print(f"avg latency: {(time.perf_counter()-t0)/N*1000:.1f} ms / вызов")

avg latency: 21.8 ms / вызов


## Как встроить в наш пайплайн

```
если (predict_endpoint(turn_audio) > 0.9 И тишина > 1с)
ИЛИ тишина > 5с
→ отправляем в LLM
```

Модель зовём на накопленном аудио текущей реплики (хвост до 8 сек),
параллельно с ASR — транскрипт ждать не нужно.

### Следующий шаг
Заменить gTTS на реальные записи звонков, разметить complete/incomplete,
построить confusion matrix и подобрать порог (см. `docs/05_metrics.md`).